In [ ]:
#| default_exp runs


# runs

> Where a training run's numbers live, and the page you read them on.

A run is a folder: `config.json`, `metrics.json`, `errors.jsonl`, and a `report.html` that opens from
a file path with no server and no network. `runs/index.html` lists every run in the store.

No sklearn in here. A run is JSON and a page, so the report can be built and tested without the
`pii` extra installed.

The report is three things a metric table cannot do: the bias sweep, so you can see what precision
a point of recall costs; per-kind precision and recall, so a kind nobody labelled enough of shows up
as one; and every mistake with the characters either side of it, filterable, because that is what
tells you whether the label was wrong or the model was.


In [ ]:
#| export
from __future__ import annotations
import html, json, os, time
from pathlib import Path

from fastcore.all import AttrDict, L


In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp


## The store


In [ ]:
#| export
def runs_home(dest=None) -> Path:
    "Where runs are written: `dest`, else `$ANYA_HOME/pii/runs`, else `~/.anya/pii/runs`."
    if dest is not None: return Path(dest).expanduser()
    return Path(os.environ.get('ANYA_HOME', Path.home()/'.anya')).expanduser()/'pii'/'runs'

def new_run(name:str=None, dest=None) -> Path:
    "An empty run folder, named `<timestamp>-<name>`, created."
    stem = time.strftime('%Y%m%d-%H%M%S') + (f'-{_slug(name)}' if name else '')
    p = runs_home(dest)/stem
    for i in range(1, 100):
        if not p.exists(): break
        p = runs_home(dest)/f'{stem}-{i}'
    p.mkdir(parents=True); return p

def _slug(s) -> str:
    "A name safe for a folder."
    return ''.join(c if c.isalnum() or c in '-_' else '-' for c in str(s)).strip('-')[:40] or 'run'

class Run(AttrDict):
    "One training run: its config, its metrics, and the report they render as."
    @property
    def errors(self) -> list: return list(self.get('metrics', {}).get('errors') or [])
    def _repr_markdown_(self):
        return summary_md(self)
    def _repr_html_(self):
        return report_html(self, standalone=False)

def save_run(path,             # a folder, from `new_run`
             config:dict,      # what was fitted, and on what
             metrics:dict,     # what `pii.evaluate` returned
             sweep:list=None,  # what `pii.tune_bias` returned
             compare:dict=None,# what `pii.compare` returned, per level
             baseline:dict=None,# the same metrics for the baseline alone
            ) -> Run:
    "Write a run and its report. Returns the `Run`."
    p = Path(path); p.mkdir(parents=True, exist_ok=True)
    errs = list(metrics.get('errors') or [])
    m = {k: v for k, v in metrics.items() if k not in ('errors', 'per_doc')}
    r = Run(path=str(p), config=dict(config), metrics=dict(m, errors=errs), sweep=list(sweep or []),
            compare=dict(compare or {}), baseline={k: v for k, v in (baseline or {}).items()
                                                  if k not in ('errors', 'per_doc')})
    (p/'config.json').write_text(json.dumps(r.config, indent=1, default=str))
    (p/'metrics.json').write_text(json.dumps(dict(m, sweep=r.sweep, compare=r.compare,
                                                  baseline=r.baseline), indent=1, default=str))
    (p/'errors.jsonl').write_text(''.join(json.dumps(e, default=str) + '\n' for e in errs))
    write_report(r)
    write_index(p.parent)
    return r

def load_run(path) -> Run:
    "A run written by `save_run`."
    p = Path(path)
    if not (p/'metrics.json').exists(): raise FileNotFoundError(f'no run at {p}')
    m = json.loads((p/'metrics.json').read_text())
    errs = [json.loads(l) for l in (p/'errors.jsonl').read_text().splitlines() if l.strip()] \
        if (p/'errors.jsonl').exists() else []
    return Run(path=str(p), config=json.loads((p/'config.json').read_text()),
               metrics=dict({k: v for k, v in m.items() if k not in ('sweep', 'compare', 'baseline')},
                            errors=errs),
               sweep=m.get('sweep') or [], compare=m.get('compare') or {}, baseline=m.get('baseline') or {})

def runs(dest=None) -> L:
    "Every run in the store, newest first."
    d = runs_home(dest)
    if not d.exists(): return L()
    return L(sorted((x for x in d.iterdir() if (x/'metrics.json').exists()), reverse=True)).map(load_run)


## Numbers, as text


In [ ]:
#| export
def summary_md(run) -> str:
    "The run as a markdown table, which is what a notebook shows without opening the report."
    m, c = run.get('metrics', {}), run.get('config', {})
    s, d = m.get('spans', {}), m.get('doc', {})
    b = (run.get('baseline') or {}).get('spans', {})
    rows = [('spans', s), ('spans, baseline', b), ('documents', d)]
    out = [f"**{c.get('name') or Path(run.get('path', 'run')).name}** — {c.get('n_train', 0)} train, "
           f"{c.get('n_valid', 0)} valid, {len(m.get('kinds') or [])} kinds, "
           f"bias {m.get('bias', 0):g}, {m.get('ms_per_doc', 0):g} ms/doc", '',
           '| | precision | recall | F1 | tp | fp | fn |', '|---|---|---|---|---|---|---|']
    for nm, r in rows:
        if not r: continue
        out.append(f"| {nm} | {r.get('precision', 0):.3f} | {r.get('recall', 0):.3f} | "
                   f"{r.get('f1', 0):.3f} | {r.get('tp', 0)} | {r.get('fp', 0)} | {r.get('fn', 0)} |")
    for lvl, cp in (run.get('compare') or {}).items():
        verdict = f"{cp['delta']:+.3f} [{cp['lo']:+.3f}, {cp['hi']:+.3f}]" if cp.get('difference') \
            else f"no difference ({cp['delta']:+.3f} [{cp['lo']:+.3f}, {cp['hi']:+.3f}])"
        out.append(f"\nF1 against the baseline, {lvl}: {verdict}")
    return '\n'.join(out)


In [ ]:
#| hide
_r = Run(path='/tmp/x', config=dict(name='t', n_train=10, n_valid=4),
         metrics=dict(spans=dict(precision=1, recall=.5, f1=.667, tp=1, fp=0, fn=1), doc={}, kinds=['a'],
                      bias=0, ms_per_doc=1.2, errors=[]),
         compare=dict(spans=dict(delta=.2, lo=.1, hi=.3, difference=True)), baseline={}, sweep=[])
test_eq('| spans | 1.000 | 0.500 | 0.667 | 1 | 0 | 1 |' in summary_md(_r), True)
test_eq('no difference' in summary_md(_r), False)


## The page

Inline SVG and one `<style>` block, so the file opens from disk with nothing fetched. Colours are the
three-slot categorical palette, which clears the colour-vision and contrast gates in both light and
dark; the sweep and the per-kind charts carry direct labels and a table twin, so no value is only
reachable by hovering or by telling two hues apart.


In [ ]:
#| export
#: Categorical slots 1-3, light and dark. Validated as a set: worst all-pairs CVD separation 9.2
#: light and 9.4 dark on the OKLab hundred-point scale, against a floor of 8.
SERIES = (('#2a78d6', '#3987e5'), ('#eb6834', '#d95926'), ('#1baf7a', '#199e70'))
#: How far the confusion cells mix slot 1 into the surface. One hue, and the surface it recedes
#: towards is the one it is drawn on, so the dark ramp is not the light ramp inverted by accident.
HEAT = (8, 18, 30, 42, 54, 66, 78)

_CSS = """
.anya-viz{--surface-1:#fcfcfb;--page:#f9f9f7;--ink:#0b0b0b;--ink-2:#52514e;--muted:#898781;
 --grid:#e1e0d9;--axis:#c3c2b7;--border:rgba(11,11,11,.10);--good:#006300;--bad:#d03b3b;
 --s1:#2a78d6;--s2:#eb6834;--s3:#1baf7a;color-scheme:light;
 font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif;color:var(--ink);background:var(--page);
 padding:24px;margin:0 auto;max-width:860px}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme=light])) .anya-viz{
 --surface-1:#1a1a19;--page:#0d0d0d;--ink:#fff;--ink-2:#c3c2b7;--grid:#2c2c2a;--axis:#383835;
 --border:rgba(255,255,255,.10);--good:#0ca30c;--s1:#3987e5;--s2:#d95926;--s3:#199e70;color-scheme:dark}}
:root[data-theme=dark] .anya-viz{--surface-1:#1a1a19;--page:#0d0d0d;--ink:#fff;--ink-2:#c3c2b7;
 --grid:#2c2c2a;--axis:#383835;--border:rgba(255,255,255,.10);--good:#0ca30c;--s1:#3987e5;
 --s2:#d95926;--s3:#199e70;color-scheme:dark}
.anya-viz h1{font-size:20px;margin:0 0 2px}
.anya-viz h2{font-size:15px;font-weight:600;margin:28px 0 2px}
.anya-viz p.sub,.anya-viz .cap{color:var(--ink-2);margin:0 0 8px}
.anya-viz .card{background:var(--surface-1);border:1px solid var(--border);border-radius:8px;
 padding:16px;margin:10px 0}
.anya-viz .tiles{display:flex;flex-wrap:wrap;gap:10px;margin:12px 0}
.anya-viz .tile{background:var(--surface-1);border:1px solid var(--border);border-radius:8px;
 padding:12px 16px;min-width:132px}
.anya-viz .tile .k{color:var(--ink-2);font-size:12px}
.anya-viz .tile .v{font-size:24px;font-weight:600}
.anya-viz .tile .d{font-size:12px;color:var(--ink-2)}
.anya-viz .up{color:var(--good)}.anya-viz .dn{color:var(--bad)}
.anya-viz .legend{display:flex;gap:14px;flex-wrap:wrap;color:var(--ink-2);font-size:12px;margin:0 0 6px}
.anya-viz .legend i{width:9px;height:9px;border-radius:50%;display:inline-block;margin-right:5px}
.anya-viz table{border-collapse:collapse;width:100%;font-size:13px}
.anya-viz th{text-align:left;color:var(--ink-2);font-weight:500;border-bottom:1px solid var(--axis);
 padding:5px 8px;position:sticky;top:0;background:var(--surface-1)}
.anya-viz td{padding:5px 8px;border-bottom:1px solid var(--grid);
 font-variant-numeric:tabular-nums;vertical-align:top}
.anya-viz td.t{font-variant-numeric:normal}
.anya-viz mark{background:#fab219;color:#0b0b0b;border-radius:2px;padding:0 1px}
.anya-viz details{margin-top:8px;font-size:13px;color:var(--ink-2)}
.anya-viz summary{cursor:pointer}
.anya-viz .scroll{max-height:420px;overflow:auto}
.anya-viz input,.anya-viz select{font:13px system-ui,sans-serif;padding:5px 8px;border-radius:6px;
 border:1px solid var(--axis);background:var(--surface-1);color:var(--ink)}
.anya-viz .bar{height:100%}
.anya-viz .cell{text-align:center;border-radius:4px;padding:10px 4px;font-variant-numeric:tabular-nums}
"""

def _esc(s) -> str: return html.escape(str(s), quote=True)

def _heat(share:float) -> str:
    "A cell's fill for `share` of the whole, as one hue mixed into the surface under it."
    pct = HEAT[min(len(HEAT) - 1, max(0, int(share*len(HEAT))))]
    return (f'background:var(--surface-1);'
            f'background:color-mix(in oklab, var(--s1) {pct}%, var(--surface-1));color:var(--ink)')

def _f(x, n=3) -> str:
    "A number, or an em space where there is none."
    return ' ' if x is None else f'{float(x):.{n}f}'


In [ ]:
#| export
def tile(label:str, value:str, note:str=None, tone:str=None) -> str:
    "One stat tile: what it is, the number, and an optional signed note."
    cls = f' class="{tone}"' if tone else ''
    n = f'<div class="d"{cls}>{_esc(note)}</div>' if note else ''
    return f'<div class="tile"><div class="k">{_esc(label)}</div><div class="v">{_esc(value)}</div>{n}</div>'

def legend(names) -> str:
    "A legend row. Present whenever two or more series are on one plot."
    return ('<div class="legend">' + ''.join(
        f'<span><i style="background:var(--s{i+1})"></i>{_esc(n)}</span>' for i, n in enumerate(names))
        + '</div>')

def _hbar(x, y, w, h, r=4) -> str:
    "A horizontal bar, rounded at the data end and square on the baseline."
    r = min(r, max(w, 0.01), h/2)
    return (f'M{x:.1f},{y:.1f} H{x+w-r:.1f} a{r:.1f},{r:.1f} 0 0 1 {r:.1f},{r:.1f} '
            f'V{y+h-r:.1f} a{r:.1f},{r:.1f} 0 0 1 {-r:.1f},{r:.1f} H{x:.1f} Z')

def svg_bars(rows,                 # `(label, [value per series], note)` per category
             names,                # series names, up to three
             gutter:int=170,       # room for the category labels
             w:int=640,            # total width
             bar:int=10,           # bar thickness, capped well under the 24px limit
            ) -> str:
    "Grouped horizontal bars on a 0-1 scale, one group per category, value at every tip."
    n, right = len(names), 64
    pw, rowh = w - gutter - right, n*(bar + 2) + 10
    h = len(rows)*rowh + 30
    o = [f'<svg viewBox="0 0 {w} {h}" width="100%" height="{h}" role="img">']
    for t in (0, .25, .5, .75, 1):                       # hairline grid, solid, one step off surface
        x = gutter + t*pw
        o.append(f'<line x1="{x:.1f}" y1="0" x2="{x:.1f}" y2="{h-30}" stroke="var(--grid)" stroke-width="1"/>')
        o.append(f'<text x="{x:.1f}" y="{h-14}" fill="var(--muted)" font-size="11" text-anchor="middle">{t:g}</text>')
    for i, (lab, vals, note) in enumerate(rows):
        y0 = i*rowh + 5
        o.append(f'<text x="{gutter-8}" y="{y0+n*(bar+2)/2+1}" fill="var(--ink-2)" font-size="12" '
                 f'text-anchor="end">{_esc(lab)}</text>')
        for j, v in enumerate(vals):
            v = 0.0 if v is None else float(v)
            y = y0 + j*(bar + 2)
            if v*pw > 0.6: o.append(f'<path d="{_hbar(gutter, y, v*pw, bar)}" fill="var(--s{j+1})">'
                                    f'<title>{_esc(lab)} · {_esc(names[j])} {v:.3f}{_esc(note or "")}</title></path>')
            o.append(f'<text x="{gutter+max(v*pw,0)+6:.1f}" y="{y+bar-1}" fill="var(--ink-2)" '
                     f'font-size="11">{v:.2f}</text>')
    return ''.join(o) + '</svg>'


In [ ]:
#| export
def svg_lines(xs,                 # the x values, shared by every series
              series,             # `(name, [y per x])`, up to three
              w:int=640, h:int=190, xlabel:str='',
             ) -> str:
    "Lines on a 0-1 y scale with the series named at its own end, for a threshold sweep."
    l, r, t, b = 40, 92, 8, 46
    pw, ph = w - l - r, h - t - b
    lo, hi = (min(xs), max(xs)) if xs else (0, 1)
    sx = lambda v: l + (0 if hi == lo else (v - lo)/(hi - lo))*pw
    sy = lambda v: t + (1 - max(0.0, min(1.0, v)))*ph
    o = [f'<svg viewBox="0 0 {w} {h}" width="100%" height="{h}" role="img">']
    for g in (0, .25, .5, .75, 1):
        y = sy(g)
        o.append(f'<line x1="{l}" y1="{y:.1f}" x2="{l+pw}" y2="{y:.1f}" stroke="var(--grid)" stroke-width="1"/>')
        o.append(f'<text x="{l-6}" y="{y+4:.1f}" fill="var(--muted)" font-size="11" text-anchor="end">{g:g}</text>')
    o.append(f'<line x1="{l}" y1="{t+ph:.1f}" x2="{l+pw}" y2="{t+ph:.1f}" stroke="var(--axis)" stroke-width="1"/>')
    for v in (xs[::max(1, len(xs)//6)] if xs else []):
        o.append(f'<text x="{sx(v):.1f}" y="{h-28}" fill="var(--muted)" font-size="11" '
                 f'text-anchor="middle">{v:g}</text>')
    if xlabel: o.append(f'<text x="{l+pw/2:.1f}" y="{h-8}" fill="var(--muted)" font-size="11" '
                        f'text-anchor="middle">{_esc(xlabel)}</text>')
    for j, (nm, ys) in enumerate(series):
        pts = ' '.join(f'{sx(x):.1f},{sy(y):.1f}' for x, y in zip(xs, ys))
        o.append(f'<polyline points="{pts}" fill="none" stroke="var(--s{j+1})" stroke-width="2" '
                 f'stroke-linejoin="round" stroke-linecap="round"/>')
        for x, y in zip(xs, ys):
            o.append(f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="7" fill="transparent">'
                     f'<title>{_esc(nm)} {y:.3f} at {x:g}</title></circle>')
        if ys:
            o.append(f'<circle cx="{sx(xs[-1]):.1f}" cy="{sy(ys[-1]):.1f}" r="4" fill="var(--s{j+1})" '
                     f'stroke="var(--surface-1)" stroke-width="2"/>')
            o.append(f'<text x="{l+pw+9}" y="{sy(ys[-1])+4:.1f}" fill="var(--ink-2)" font-size="11">'
                     f'{_esc(nm)}</text>')
    return ''.join(o) + '</svg>'

def table(head, rows, cls='') -> str:
    "A plain table. Every chart has one, so nothing is only readable as a colour."
    h = ''.join(f'<th>{_esc(c)}</th>' for c in head)
    b = ''.join('<tr>' + ''.join(f'<td>{c}</td>' for c in r) + '</tr>' for r in rows)
    return f'<table class="{cls}"><thead><tr>{h}</tr></thead><tbody>{b}</tbody></table>'


In [ ]:
#| export
_JS = """
(function(){
 var box=document.getElementById('q'), sel=document.getElementById('why'),
     rows=[].slice.call(document.querySelectorAll('tr[data-why]'));
 function go(){var q=(box.value||'').toLowerCase(), w=sel.value;
  rows.forEach(function(r){
   var okw = !w || r.getAttribute('data-why')===w,
       okq = !q || r.textContent.toLowerCase().indexOf(q)>=0;
   r.style.display = (okw&&okq) ? '' : 'none';});}
 box.addEventListener('input',go); sel.addEventListener('change',go);
})();
"""

def confusion_html(doc:dict, n:int) -> str:
    "The document decision as a two by two, shaded by share and with the count in every cell."
    tp, fp, fn, tn = (int(doc.get(k) or 0) for k in ('tp', 'fp', 'fn', 'tn'))
    def cell(v):
        return f'<div class="cell" style="{_heat(v/max(n, 1))}">{v}</div>'
    grid = ('<div style="display:grid;grid-template-columns:120px 1fr 1fr;gap:2px;align-items:center">'
            f'<div></div><div class="cap" style="text-align:center">said PII</div>'
            f'<div class="cap" style="text-align:center">said clean</div>'
            f'<div class="cap">is PII</div>{cell(tp)}{cell(fn)}'
            f'<div class="cap">is clean</div>{cell(fp)}{cell(tn)}</div>')
    scale = ('<div class="cap" style="margin-top:6px">share of ' + str(n) + ' documents, none to all '
             + ''.join(f'<span style="display:inline-block;width:14px;height:9px;{_heat(i/(len(HEAT)-1))}">'
                       '</span>' for i in range(len(HEAT))) + '</div>')
    return grid + scale

def errors_html(errs:list, limit:int=300) -> str:
    "Every mistake with the characters either side of it, filterable by kind and by what went wrong."
    whys = sorted({e.get('why', '') for e in errs})
    opts = ''.join(f'<option value="{_esc(w)}">{_esc(w)}</option>' for w in whys)
    rows = []
    for e in errs[:limit]:
        ctx = (f'{_esc(e.get("before", ""))}<mark>{_esc(e.get("text", ""))}</mark>'
               f'{_esc(e.get("after", ""))}') if e.get('text') else _esc((e.get('full') or '')[:160])
        rows.append(f'<tr data-why="{_esc(e.get("why", ""))}"><td>{e.get("i", "")}</td>'
                    f'<td class="t">{_esc(e.get("why", ""))}</td><td class="t">{_esc(e.get("kind") or "")}</td>'
                    f'<td class="t">{ctx}</td></tr>')
    if not rows: return '<p class="cap">Nothing was missed and nothing was invented.</p>'
    return ('<div style="display:flex;gap:8px;margin-bottom:8px">'
            '<input id="q" placeholder="filter the text" style="flex:1"/>'
            f'<select id="why"><option value="">every kind of mistake</option>{opts}</select></div>'
            '<div class="scroll"><table><thead><tr><th>doc</th><th>what</th><th>kind</th>'
            f'<th>where</th></tr></thead><tbody>{"".join(rows)}</tbody></table></div>'
            + (f'<p class="cap">{len(errs)} shown of what was kept.</p>' if len(errs) > limit else ''))


In [ ]:
#| export
def report_html(run, standalone:bool=True) -> str:
    "The whole run as one page: the headline numbers, the sweep, per-kind scores, and every mistake."
    m, c = run.get('metrics', {}) or {}, run.get('config', {}) or {}
    s, d = m.get('spans', {}) or {}, m.get('doc', {}) or {}
    bl = (run.get('baseline') or {}).get('spans', {}) or {}
    cp = (run.get('compare') or {}).get('spans', {}) or {}
    name = c.get('name') or Path(run.get('path', 'run')).name
    tone = None if not cp else ('up' if cp.get('difference') and cp.get('delta', 0) > 0 else
                                'dn' if cp.get('difference') and cp.get('delta', 0) < 0 else None)
    note = (f"{cp['delta']:+.3f} vs baseline [{cp['lo']:+.3f}, {cp['hi']:+.3f}]"
            if cp.get('difference') else
            f"no difference vs baseline [{cp.get('lo', 0):+.3f}, {cp.get('hi', 0):+.3f}]" if cp else None)
    o = [f'<h1>{_esc(name)}</h1>',
         f'<p class="sub">{c.get("n_train", 0)} train, {c.get("n_valid", 0)} valid, '
         f'{len(m.get("kinds") or [])} kinds learned, mode {_esc(m.get("mode", ""))}, '
         f'bias {m.get("bias", 0):g}{", exact offsets" if not m.get("relaxed") else ", overlap counted"}'
         f' &middot; {_esc(c.get("fitted_at", ""))}</p>',
         '<div class="tiles">',
         tile('Span F1', _f(s.get('f1')), note, tone),
         tile('Span precision', _f(s.get('precision')), f"{s.get('fp', 0)} invented"),
         tile('Span recall', _f(s.get('recall')), f"{s.get('fn', 0)} missed"),
         tile('Document F1', _f(d.get('f1')), f"{_f(d.get('accuracy'))} accuracy"),
         tile('Baseline F1', _f(bl.get('f1')), 'patterns alone'),
         tile('Per document', f"{m.get('ms_per_doc', 0):g} ms", 'both layers'),
         '</div>']
    by = m.get('by_kind') or {}
    if by:
        rows = sorted(by.items(), key=lambda t: -t[1].get('support', 0))
        o += ['<h2>Precision and recall, per kind</h2>',
              '<p class="cap">A kind with a low bar and a high support is one the model has examples '
              'of and still gets wrong. A kind with no bar at all was never learned.</p>',
              '<div class="card">', legend(['precision', 'recall']),
              svg_bars([(f'{k} ({v.get("support", 0)})', [v.get('precision'), v.get('recall')], '')
                        for k, v in rows], ['precision', 'recall']),
              '<details><summary>as a table</summary>',
              table(['kind', 'precision', 'recall', 'F1', 'tp', 'fp', 'fn', 'support'],
                    [[k, _f(v.get('precision')), _f(v.get('recall')), _f(v.get('f1')), v.get('tp', 0),
                      v.get('fp', 0), v.get('fn', 0), v.get('support', 0)] for k, v in rows]),
              '</details></div>']
    sw = run.get('sweep') or []
    if sw:
        xs = [r['bias'] for r in sw]
        o += ['<h2>What a point of recall costs</h2>',
              '<p class="cap">The decoding bias, swept on a split held out of the training half. '
              f'The fit kept {m.get("bias", 0):g}'
              + ('.' if run.get('config', {}).get('tuned') else ', which the sweep did not choose: '
                 'the argmax of a small split does not transfer.') + '</p>',
              '<div class="card">', legend(['precision', 'recall', 'F1']),
              svg_lines(xs, [('precision', [r['precision'] for r in sw]),
                             ('recall', [r['recall'] for r in sw]),
                             ('F1', [r['f1'] for r in sw])], xlabel='bias on O'),
              '<details><summary>as a table</summary>',
              table(['bias', 'precision', 'recall', 'F1'],
                    [[f"{r['bias']:g}", _f(r['precision']), _f(r['recall']), _f(r['f1'])] for r in sw]),
              '</details></div>']
    if d:
        o += ['<h2>The document decision</h2>',
              '<p class="cap">Whether the document is somebody&rsquo;s business, which is the call a '
              'retrieval gate makes. Spans and the document classifier are OR&rsquo;d.</p>',
              '<div class="card">', confusion_html(d, m.get('n', 0)),
              f'<p class="cap" style="margin-top:10px">precision {_f(d.get("precision"))}, recall '
              f'{_f(d.get("recall"))}, F1 {_f(d.get("f1"))}'
              + (f', class accuracy {_f(m.get("class_accuracy"))}' if m.get('class_accuracy') is not None else '')
              + '</p></div>']
    o += ['<h2>Every mistake</h2>', '<div class="card">', errors_html(m.get('errors') or []), '</div>',
          '<details><summary>the fit, in full</summary><pre style="white-space:pre-wrap">'
          f'{_esc(json.dumps(c, indent=1, default=str))}</pre></details>']
    body = ''.join(o)
    if not standalone: return f'<style>{_CSS}</style><div class="anya-viz">{body}</div>'
    return (f'<!doctype html><html lang="en"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width,initial-scale=1">'
            f'<title>{_esc(name)}</title><style>{_CSS}</style></head><body>'
            f'<div class="anya-viz">{body}</div><script>{_JS}</script></body></html>')

def write_report(run, path=None) -> Path:
    "Write `report.html` into the run's folder."
    p = Path(path or Path(run['path'])/'report.html')
    p.write_text(report_html(run)); return p


In [ ]:
#| export
def index_html(rs) -> str:
    "Every run in the store, newest first, each linking to its own report."
    rows = []
    for r in rs:
        m, c = r.get('metrics', {}), r.get('config', {})
        s, d = m.get('spans', {}) or {}, m.get('doc', {}) or {}
        nm = Path(r['path']).name
        rows.append([f'<a href="{_esc(nm)}/report.html">{_esc(c.get("name") or nm)}</a>',
                     c.get('n_train', 0), c.get('n_valid', 0), len(m.get('kinds') or []),
                     _f(s.get('precision')), _f(s.get('recall')), _f(s.get('f1')), _f(d.get('f1')),
                     _esc(c.get('fitted_at', ''))])
    body = ('<h1>anya pii runs</h1><p class="sub">' + str(len(rows)) + ' runs, newest first</p>'
            '<div class="card">' + (table(['run', 'train', 'valid', 'kinds', 'precision', 'recall',
                                           'span F1', 'doc F1', 'fitted'], rows)
                                    if rows else '<p class="cap">No runs yet.</p>') + '</div>')
    return (f'<!doctype html><html lang="en"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width,initial-scale=1"><title>anya pii runs</title>'
            f'<style>{_CSS}</style></head><body><div class="anya-viz">{body}</div></body></html>')

def write_index(dest=None) -> Path:
    "Write `index.html` over the whole store."
    d = runs_home(dest); d.mkdir(parents=True, exist_ok=True)
    p = d/'index.html'; p.write_text(index_html(runs(d))); return p

def log_wandb(run, project:str='anya-pii', **kw) -> str|None:
    "Log one run to wandb when it is installed, and return its URL. `None` when it is not."
    try: import wandb
    except ImportError: return None
    m = run.get('metrics', {})
    flat = {f'{g}/{k}': v for g in ('spans', 'doc') for k, v in (m.get(g) or {}).items()
            if isinstance(v, (int, float))}
    w = wandb.init(project=project, name=(run.get('config') or {}).get('name') or Path(run['path']).name,
                   config=dict(run.get('config') or {}), **kw)
    w.log(flat | {'ms_per_doc': m.get('ms_per_doc', 0)})
    for r in (run.get('sweep') or []): w.log({f'sweep/{k}': v for k, v in r.items()})
    url = getattr(w, 'url', None)
    w.finish()
    return url


In [ ]:
#| hide
_d = Path(mkdtemp())
_p = new_run('demo', _d)
_r2 = save_run(_p, config=dict(name='demo', n_train=8, n_valid=4, fitted_at='now'),
               metrics=dict(spans=dict(precision=.9, recall=.8, f1=.85, tp=8, fp=1, fn=2), n=4,
                            doc=dict(precision=1., recall=.75, f1=.857, tp=3, fp=0, fn=1, tn=0, accuracy=.75),
                            by_kind=dict(emp_id=dict(precision=1., recall=.9, f1=.95, tp=9, fp=0, fn=1, support=10)),
                            kinds=['emp_id'], mode='hybrid', bias=0.5, ms_per_doc=1.1, relaxed=False,
                            errors=[dict(i=1, why='missed', kind='emp_id', text='EMP-1', before='by ', after=' ok')]),
               sweep=[dict(bias=0.0, precision=.8, recall=.9, f1=.85), dict(bias=1.0, precision=.9, recall=.8, f1=.85)],
               compare=dict(spans=dict(delta=.3, lo=.1, hi=.5, difference=True)),
               baseline=dict(spans=dict(precision=1., recall=.2, f1=.333, tp=2, fp=0, fn=8)))
_h = (_p/'report.html').read_text()
test_eq(_h.startswith('<!doctype html>'), True)
test_eq('<svg' in _h and 'EMP-1' in _h, True)
test_eq('+0.300 vs baseline' in _h, True)
test_eq((_d/'index.html').exists(), True)
test_eq('demo/report.html' in (_d/'index.html').read_text(), True)
test_eq(len(runs(_d)), 1)
test_eq(load_run(_p).metrics['spans']['f1'], .85)
test_eq(log_wandb(_r2), None)                             # not installed here, and not an error
test_fail(lambda: load_run(_d/'nope'), contains='no run at')
# a run with nothing in it still renders: the report is what you look at when a fit went wrong
test_eq('<h1>' in report_html(Run(path=str(_p), metrics={}, config={})), True)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
